In [ ]:
import geopandas as gpd
import pandas as pd
from useful_functions import *
import pickle
import time
import resource_bifiltration_v3 as RB
import importlib
import os 
from persim import plot_diagrams
import random
from ripser import ripser
from shapely.geometry import Polygon, LineString, Point, shape
import matplotlib as mpl

# importlib.reload(RB)
from geopy import distance

import sys
from contextlib import redirect_stdout
from datetime import datetime
import numpy as np
import networkx as nx

plt.rc('text', usetex=True)
plt.rc('font', family='serif')

# LANDFILLS

In [ ]:
results_folder = 'Results_Landfills_Nuisance'
blocks = gpd.read_file('geographic_data/tl_2020_17_tract/tl_2020_17_tract.shp')
blocks = blocks.drop(index=[3235,2385])
blocks = blocks.reset_index(drop=True)
blocks = blocks.to_crs('EPSG:26916')

D = np.load('geographic_data/landfills_distances_geodesic_NEW.npy')

blocks['mindist'] = np.min(D, axis=1)

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

rs_all = np.arange(10, 41, 2.5)
rs = rs_all[::2]
rs = rs[::2]

fig, axes = plt.subplots(1,len(rs), figsize=(len(rs)*3,5))

for t, ax in zip(rs, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)

    blocks_t = blocks[blocks['mindist'] <= t]
    blocks_t.plot(ax=ax, color='C5')

    
    ax.set_title(f'Filtration Value: \n {int(t)} kilometers', fontsize=20)

    # df.plot(ax=ax, color='C1', marker='.', zorder=1000, markersize=50, alpha=0.5)
    ax.axis('off')


plt.tight_layout()

plt.savefig(f'{results_folder}/VR_Filtration_baseline_Sublevel.png', dpi=300, bbox_inches='tight')

In [ ]:
G = adjacency_graph(blocks, calc_area=True)
for n in G:
    G.nodes[n]['weight'] = min(D[n,:])
    centroid = G.nodes[n]['geometry'].centroid
    G.nodes[n]['point'] = [centroid.y, centroid.x]

In [ ]:
for e in G.edges():
    G.edges[e[:2]]['weight'] = max( [G.nodes[e[0]]['weight'], G.nodes[e[1]]['weight']] )

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

rs = np.arange(10, 41, 5)
# rs = rs[::2]

fig, axes = plt.subplots(1,len(rs), figsize=(len(rs)*3,5))


for t, ax in zip(rs, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)
    
    keep = [(u, v) for u, v, d in G.edges(data=True) if d.get('weight') <= t]
    H = G.edge_subgraph(keep).copy()  # H has only those edges (and their incident nodes)
    comps = [c for c in nx.connected_components(H)]

    for c in comps:
        pos = {j: G.nodes[j]['point'][::-1] for j in c}
        nx.draw_networkx_edges(G.subgraph(list(c)), pos=pos, ax=ax, alpha=0.75, edge_color='C5')

    # df.plot(ax=ax, color='C0', marker='.', zorder=1000, markersize=50, alpha=0.5)
    ax.set_title(f'Filtration Value: \n {int(t)} kilometers', fontsize=20)
    ax.axis('off')

plt.tight_layout()

plt.savefig(f'{results_folder}/VR_Filtration_baseline_network.png', dpi=300, bbox_inches='tight')

In [ ]:
G = adjacency_graph(blocks, calc_area=True)

for n in G:
    G.nodes[n]['weight'] = min(D[n,:])
    centroid = G.nodes[n]['geometry'].centroid
    G.nodes[n]['point'] = [centroid.y, centroid.x]

total_area = sum([ G.nodes[n]['area'] for n in G])

for e in G.edges():
    G.edges[e[:2]]['weight'] = min( [G.nodes[e[0]]['weight'], G.nodes[e[1]]['weight']] )

In [ ]:
threshold = 0.1
all_comps = []
all_comps_values = []
for t, ax in zip(rs_all, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)
    
    keep = [(u, v) for u, v, d in G.edges(data=True) if d.get('weight') <= t]
    H = G.edge_subgraph(keep).copy()  # H has only those edges (and their incident nodes)
    comps = [c for c in nx.connected_components(H)]

    for c in comps:
        if sum([G.nodes[j]['area'] for j in c])/total_area < threshold:
            all_comps.append(list(c))
            all_comps_values.append((set(c),t))

In [ ]:
all_comps.sort(key=len)

maxed_out_comps = []
for i, c1 in enumerate(all_comps):
    not_subset = True
    for j, c2 in enumerate(all_comps):
        if i!=j:
            if set(c1).issubset(set(c2)):
                not_subset = False
    
    if not_subset:
        maxed_out_comps.append(c1)

In [ ]:
maxed_out_comps_value = []
for c in maxed_out_comps:
    for (c_new, v) in all_comps_values:
        if set(c_new) == set(c):
            maxed_out_comps_value.append(v)

In [ ]:
# ########## All Maximal Components, colored by range ##########
# cmap = mpl.colormaps['Oranges_r']
# norm = mpl.colors.Normalize(vmin=0, vmax=1)  # scale from 0 to 1

# union_geom = blocks.union_all()
# boundary = gpd.GeoSeries(union_geom.boundary)

# fig, ax = plt.subplots(figsize=(5,5))

# boundary.plot(ax=ax, color='black', linewidth=1)

# for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
#     color = np.where(rs_all == c_color)[0][0] #random.randint(0,9)
#     color = color/len(rs_all)

#     blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
#     blocks[blocks['keep?']==1].plot(color=cmap(color), ax=ax) #, norm=norm)

# plt.axis('off')

# plt.savefig(f'{results_folder}/VR_components_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
# ########## All Maximal Components, colored by range ##########
# cmap = mpl.colormaps['viridis']
# norm = mpl.colors.Normalize(vmin=0, vmax=1)  # scale from 0 to 1

# union_geom = blocks.union_all()
# boundary = gpd.GeoSeries(union_geom.boundary)

# fig, ax = plt.subplots(figsize=(5,5))

# boundary.plot(ax=ax, color='black', linewidth=1)

# for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
#     color = np.where(rs_all == c_color)[0][0] #random.randint(0,9)
#     color = color/len(rs_all)

#     blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
#     blocks[blocks['keep?']==1].plot(color=cmap(color), ax=ax) #, norm=norm)

# plt.axis('off')

# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])  # required for older versions of matplotlib
# cbar = fig.colorbar(sm, ax=ax)
# cbar.set_label('Range')

# plt.savefig(f'{results_folder}/VR_components_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
########## All Maximal Components, colored randomly ##########
cmap = mpl.colormaps['viridis']

union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

fig, ax = plt.subplots(figsize=(5,5))

boundary.plot(ax=ax, color='black', linewidth=1)

for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
    color = random.randint(0,9)
    # color = color/len(rs_all)

    blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
    blocks[blocks['keep?']==1].plot(color=f'C{color}', ax=ax) #, norm=norm)

plt.axis('off')

plt.savefig(f'{results_folder}/VR_components_baseline_randcolor.png', dpi=300, bbox_inches='tight')

In [ ]:
# fig, axes = plt.subplots(4,5, figsize=(10,10))
# axes = axes.flatten()
# for c, c_color,ax in zip(maxed_out_comps, maxed_out_comps_value,axes):
#     boundary.plot(ax=ax, color='black', linewidth=1)

#     blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
#     blocks[blocks['keep?']==1].plot(color=cmap(color), ax=ax) #, norm=norm)

#     ax.axis('off')

# plt.tight_layout()

# PUBS -  NUISANCE

In [ ]:
results_folder = 'Results_Pubs_Nuisance'

In [ ]:
blocks = gpd.read_file('geographic_data/chicago_blocks.shp')
blocks = blocks.to_crs('EPSG:26916')
D = np.load('geographic_data/pubs_distances_L2_NEW.npy')

blocks['mindist'] = np.min(D, axis=1)
        
G = adjacency_graph(blocks, calc_area=True)

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

rs_all = np.arange(200, 2001, 150)
# rs_all = np.array([int(r) for r in rs_all])
rs = rs_all[::2]
rs = rs[::2]

fig, axes = plt.subplots(1,len(rs), figsize=(len(rs)*3,5))

for t, ax in zip(rs, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)

    blocks_t = blocks[blocks['mindist'] <= t]
    blocks_t.plot(ax=ax, color='C3')

    
    ax.set_title(f'Filtration Value: \n {int(t)} meters', fontsize=20)

    ax.axis('off')


plt.tight_layout()

plt.savefig(f'{results_folder}/VR_Filtration_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
G = adjacency_graph(blocks, calc_area=True)

for n in G:
    G.nodes[n]['weight'] = min(D[n,:])
    centroid = G.nodes[n]['geometry'].centroid
    G.nodes[n]['point'] = [centroid.y, centroid.x]

total_area = sum([ G.nodes[n]['area'] for n in G])

for e in G.edges():
    G.edges[e[:2]]['weight'] = min( [G.nodes[e[0]]['weight'], G.nodes[e[1]]['weight']] )

In [ ]:
threshold = 0.2
all_comps = []
all_comps_values = []
for t, ax in zip(rs_all, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)
    
    keep = [(u, v) for u, v, d in G.edges(data=True) if d.get('weight') <= t]
    H = G.edge_subgraph(keep).copy()  # H has only those edges (and their incident nodes)
    comps = [c for c in nx.connected_components(H)]

    for c in comps:
        if sum([G.nodes[j]['area'] for j in c])/total_area < threshold:
            all_comps.append(list(c))
            all_comps_values.append((set(c),t))

In [ ]:
all_comps.sort(key=len)

maxed_out_comps = []
for i, c1 in enumerate(all_comps):
    not_subset = True
    for j, c2 in enumerate(all_comps):
        if i!=j:
            if set(c1).issubset(set(c2)):
                not_subset = False
    
    if not_subset:
        maxed_out_comps.append(c1)

In [ ]:
maxed_out_comps_value = []
for c in maxed_out_comps:
    for (c_new, v) in all_comps_values:
        if set(c_new) == set(c):
            maxed_out_comps_value.append(v)

In [ ]:
# cmap = mpl.colormaps['Reds_r']
# norm = mpl.colors.Normalize(vmin=0, vmax=1)  # scale from 0 to 1


# union_geom = blocks.union_all()
# boundary = gpd.GeoSeries(union_geom.boundary)

# fig, ax = plt.subplots(figsize=(5,5))

# boundary.plot(ax=ax, color='black', linewidth=1)

# for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
#     color = np.where(rs_all == c_color)[0][0] #random.randint(0,9)
#     color = color/len(rs_all)
    
#     blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
#     blocks[blocks['keep?']==1].plot(color=cmap(color), ax=ax)

# plt.axis('off')

# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])  # required for older versions of matplotlib
# cbar = fig.colorbar(sm, ax=ax)
# cbar.set_label('Range')

# plt.savefig(f'{results_folder}/VR_components_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

fig, ax = plt.subplots(figsize=(5,5))

boundary.plot(ax=ax, color='black', linewidth=1)

for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
    color = random.randint(0,9)
    
    blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
    blocks[blocks['keep?']==1].plot(color=f'C{color}', ax=ax)

plt.axis('off')

plt.savefig(f'{results_folder}/VR_components_baseline_randcolor.png', dpi=300, bbox_inches='tight')

# PUBS -  RESOURCE

In [ ]:
results_folder = 'Results_Pubs_Resource'

In [ ]:
blocks = gpd.read_file('geographic_data/chicago_blocks.shp')
blocks = blocks.to_crs('EPSG:26916')
D = np.load('geographic_data/pubs_distances_L1_NEW.npy')

blocks['mindist'] = np.min(D, axis=1)

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

rs_all = np.arange(200, 2001, 150)
rs_all = np.array([int(r) for r in rs_all])
rs = rs_all[::-2]
rs = rs[::2]

fig, axes = plt.subplots(1,len(rs), figsize=(len(rs)*3,5))

for t, ax in zip(rs, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)

    blocks_t = blocks[blocks['mindist'] >= t]
    blocks_t.plot(ax=ax, color='C0')

    
    ax.set_title(f'Filtration Value: \n {int(t)} meters', fontsize=20)

    ax.axis('off')


plt.tight_layout()

plt.savefig(f'{results_folder}/VR_Filtration_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
G = adjacency_graph(blocks, calc_area=True)

for n in G:
    G.nodes[n]['weight'] = min(D[n,:])
    centroid = G.nodes[n]['geometry'].centroid
    G.nodes[n]['point'] = [centroid.y, centroid.x]

total_area = sum([ G.nodes[n]['area'] for n in G])

for e in G.edges():
    G.edges[e[:2]]['weight'] = min( [G.nodes[e[0]]['weight'], G.nodes[e[1]]['weight']] )

In [ ]:
threshold = 0.2
all_comps = []
all_comps_values = []
for t, ax in zip(rs_all, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)
    
    keep = [(u, v) for u, v, d in G.edges(data=True) if d.get('weight') >= t]
    H = G.edge_subgraph(keep).copy()  # H has only those edges (and their incident nodes)
    comps = [c for c in nx.connected_components(H)]

    for c in comps:
        if sum([G.nodes[j]['area'] for j in c])/total_area <= threshold:
            all_comps.append(list(c))
            all_comps_values.append((set(c),t))

In [ ]:
all_comps.sort(key=len)

maxed_out_comps = []
for i, c1 in enumerate(all_comps):
    not_subset = True
    for j, c2 in enumerate(all_comps):
        if i!=j:
            if set(c1).issubset(set(c2)):
                not_subset = False
    
    if not_subset:
        maxed_out_comps.append(c1)

In [ ]:
maxed_out_comps_value = []
for c in maxed_out_comps:
    for (c_new, v) in all_comps_values:
        if set(c_new) == set(c):
            maxed_out_comps_value.append(v)

In [ ]:
# cmap = mpl.colormaps['Blues_r']
# norm = mpl.colors.Normalize(vmin=0, vmax=1)  # scale from 0 to 1

# union_geom = blocks.union_all()
# boundary = gpd.GeoSeries(union_geom.boundary)

# fig, ax = plt.subplots(figsize=(5,5))

# boundary.plot(ax=ax, color='black', linewidth=1)

# for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
#     color = np.where(rs_all == c_color)[0][0] #random.randint(0,9)
#     color = color/len(rs_all)

#     blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
#     blocks[blocks['keep?']==1].plot(color=cmap(color), ax=ax)

# plt.axis('off')

# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])  # required for older versions of matplotlib
# cbar = fig.colorbar(sm, ax=ax)
# cbar.set_label('Range')

# plt.savefig(f'{results_folder}/VR_components_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

fig, ax = plt.subplots(figsize=(5,5))

boundary.plot(ax=ax, color='black', linewidth=1)

for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
    color = random.randint(0,9)
    
    blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
    blocks[blocks['keep?']==1].plot(color=f'C{color}', ax=ax)

plt.axis('off')

plt.savefig(f'{results_folder}/VR_components_baseline_randcolor.png', dpi=300, bbox_inches='tight')

# PARKS

In [ ]:
path_to_files='geographic_data/'
results_folder = 'Results_Parks_Resource'

In [ ]:
blocks = gpd.read_file(path_to_files + 'chicago_blocks.shp')
D = np.load(path_to_files + 'parks_distances_l1.npy')
blocks['mindist'] = np.min(D, axis=1)

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

rs_all = np.arange(200, 2001, 150)
# rs = [int(r) for r in rs]
rs = rs_all[::-2]
rs = rs[::2]

fig, axes = plt.subplots(1,len(rs), figsize=(len(rs)*3,5))

for t, ax in zip(rs, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)

    blocks_t = blocks[blocks['mindist'] >= t]
    blocks_t.plot(ax=ax, color='C2')

    
    ax.set_title(f'Filtration Value: \n {int(t)} meters', fontsize=20)

    ax.axis('off')


plt.tight_layout()

plt.savefig(f'{results_folder}/VR_Filtration_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
G = adjacency_graph(blocks, calc_area=True)

for n in G:
    G.nodes[n]['weight'] = min(D[n,:])
    centroid = G.nodes[n]['geometry'].centroid
    G.nodes[n]['point'] = [centroid.y, centroid.x]

total_area = sum([ G.nodes[n]['area'] for n in G])

for e in G.edges():
    G.edges[e[:2]]['weight'] = min( [G.nodes[e[0]]['weight'], G.nodes[e[1]]['weight']] )

In [ ]:
threshold = 0.2
all_comps = []
all_comps_values = []
for t, ax in zip(rs, axes):
    boundary.plot(ax=ax, color='black', linewidth=1)
    
    keep = [(u, v) for u, v, d in G.edges(data=True) if d.get('weight') >= t]
    H = G.edge_subgraph(keep).copy()  # H has only those edges (and their incident nodes)
    comps = [c for c in nx.connected_components(H)]

    for c in comps:
        if sum([G.nodes[j]['area'] for j in c])/total_area < threshold:
            all_comps.append(list(c))
            all_comps_values.append((set(c),t))

In [ ]:
all_comps.sort(key=len)

maxed_out_comps = []
for i, c1 in enumerate(all_comps):
    not_subset = True
    for j, c2 in enumerate(all_comps):
        if i!=j:
            if set(c1).issubset(set(c2)):
                not_subset = False
    
    if not_subset:
        maxed_out_comps.append(c1)

In [ ]:
maxed_out_comps_value = []
for c in maxed_out_comps:
    for (c_new, v) in all_comps_values:
        if set(c_new) == set(c):
            maxed_out_comps_value.append(v)

In [ ]:
cmap = mpl.colormaps['Greens_r']
norm = mpl.colors.Normalize(vmin=0, vmax=1)  # scale from 0 to 1

union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

fig, ax = plt.subplots(figsize=(5,5))

boundary.plot(ax=ax, color='black', linewidth=1)

for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
    color = np.where(rs_all == c_color)[0][0] #random.randint(0,9)
    color = color/len(rs_all)
    
    blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
    blocks[blocks['keep?']==1].plot(color=cmap(color), ax=ax)

plt.axis('off')

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # required for older versions of matplotlib
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Range')

plt.savefig(f'{results_folder}/VR_components_baseline.png', dpi=300, bbox_inches='tight')

In [ ]:
union_geom = blocks.union_all()
boundary = gpd.GeoSeries(union_geom.boundary)

fig, ax = plt.subplots(figsize=(5,5))

boundary.plot(ax=ax, color='black', linewidth=1)

for c, c_color in zip(maxed_out_comps, maxed_out_comps_value):
    color = random.randint(0,9)
    
    blocks['keep?'] = [1 if i in c else 0 for i in range(len(blocks))]
    
    blocks[blocks['keep?']==1].plot(color=f'C{color}', ax=ax)

plt.axis('off')

plt.savefig(f'{results_folder}/VR_components_baseline_randcolor.png', dpi=300, bbox_inches='tight')